# Fine-Tune Open Weight Language Models for Function Calling in Strands Agents

[![License](https://img.shields.io/badge/License-Apache%202.0-blue.svg)](https://opensource.org/licenses/Apache-2.0)
[![Python 3.10+](https://img.shields.io/badge/python-3.10+-blue.svg)](https://www.python.org/downloads/)

## Introduction

This notebook demonstrates how to fine-tune small language models (1-3B parameters) for reliable function calling in edge environments. We leverage optimized training pipelines to achieve 2-5x faster training with 50% less memory usage compared to standard implementations.

Function calling, also known as tool use, enables language models to interact with external systems through structured API calls. This is critical for edge deployment where models must control physical devices, query databases, or invoke services with high reliability.

### The Problem We're Solving

Edge devices in vehicles and industrial settings need AI assistants that can:
- **Understand natural language**: "It's too hot" → climate control
- **Execute actions locally**: No cloud dependency for critical controls
- **Work with limited resources**: 2-4GB RAM, no GPU required
- **Maintain high accuracy**: Safety-critical operations demand reliability

### Our Solution: Fine-Tuned Tool Calling

Instead of using a general-purpose model, we specialize Qwen3-1.7B for specific tools:

```mermaid
graph LR
    A[User Input:<br/>'Set the temperature to 72 degrees'] --> B[Model Recognition:<br/>Intent = climate_control<br/>Parameter = 72]
    B --> C[Strands Format:<br/>toolUse.name: climate_control<br/>toolUse.input.command: 'set to 72']
    C --> D[Agent Execution:<br/>Virtual ECU updates<br/>climate state]
    D --> E[User Feedback:<br/>'Temperature set to 72°F']
    
    style A fill:#e1f5fe
    style B fill:#fff3e0
    style C fill:#f3e5f5
    style D fill:#e8f5e9
    style E fill:#fce4ec
```

### What You'll Learn

- Generate high-quality synthetic training data using teacher-student approaches
- Fine-tune models efficiently using LoRA (Low-Rank Adaptation)
- Quantize models for edge deployment while preserving accuracy
- Benchmark performance improvements systematically
- Deploy models using llama.cpp for production inference

### Understanding Model Quantization

Quantization is the process of reducing numerical precision to compress models while preserving performance. In neural networks, weights and activations typically use 32-bit (FP32) or 16-bit (FP16/BF16) floating-point numbers. Quantization reduces these to 8-bit integers or even 4-bit representations, achieving:

- **Size Reduction**: 4-bit quantization reduces model size by ~75% compared to FP16
- **Speed Improvement**: Integer operations are faster than floating-point on most hardware
- **Memory Efficiency**: Enables deployment on resource-constrained edge devices
- **Minimal Accuracy Loss**: Modern quantization methods preserve >99% of model quality

#### Quantization Methods

**K-means Quantization (K-quants)**: Groups similar weights into clusters, storing cluster indices instead of full values. The 'K' variants (Q4_K_M, Q5_K_M) use different bit allocations for different tensor components.

**Dynamic vs Static**: Dynamic quantization determines scale factors at runtime, while static uses pre-computed values. We use static for predictable edge performance.

**Mixed Precision**: Critical layers (like embeddings) maintain higher precision while less sensitive layers use aggressive quantization.

In [ ]:
%%capture
# Install dependencies
!pip install -q --upgrade pip
!pip install -q -e ../../[function-calling]

In [ ]:
# Configuration
import os

os.environ['AWS_BEARER_TOKEN_BEDROCK'] = 'YOUR_BEARER_TOKEN_HERE'
os.environ['AWS_REGION'] = 'us-east-1'

In [ ]:
# Imports
import json
import torch
from pathlib import Path
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, TaskType, get_peft_model

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

## Data Preparation

In [ ]:
# Load training data
train_path = Path('data/train.jsonl')
test_path = Path('data/test.jsonl')

train_texts = []
with open(train_path, 'r') as f:
    for line in f:
        data = json.loads(line)
        train_texts.append(data['text'])

test_texts = []
with open(test_path, 'r') as f:
    for line in f:
        data = json.loads(line)
        test_texts.append(data['text'])

print(f"Train: {len(train_texts)}, Test: {len(test_texts)}")

## Model Setup

In [ ]:
# Load base model
model_name = "Qwen/Qwen3-1.7B"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Model loaded: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B parameters")

In [ ]:
# Configure LoRA
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()
model.enable_input_require_grads()

## Training

In [ ]:
# Prepare datasets
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=1024
    )

train_dataset = Dataset.from_dict({"text": train_texts})
test_dataset = Dataset.from_dict({"text": test_texts})

tokenized_train = train_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
tokenized_test = test_dataset.map(tokenize_function, batched=True, remove_columns=["text"])

tokenized_train.set_format("torch")
tokenized_test.set_format("torch")

In [ ]:
# Training configuration
training_args = TrainingArguments(
    output_dir="./qwen3-function-calling",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    warmup_steps=100,
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    learning_rate=2e-4,
    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True,
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
)

In [ ]:
# Train the model
trainer.train()
trainer.save_model("./qwen3-function-calling-final")
tokenizer.save_pretrained("./qwen3-function-calling-final")

In [ ]:
# Display training metrics
import matplotlib.pyplot as plt

# Extract loss from training history
train_loss = [log['loss'] for log in trainer.state.log_history if 'loss' in log]
eval_loss = [log['eval_loss'] for log in trainer.state.log_history if 'eval_loss' in log]

# Create simple loss plot
if train_loss or eval_loss:
    plt.figure(figsize=(10, 5))
    
    if train_loss:
        plt.subplot(1, 2, 1)
        plt.plot(train_loss)
        plt.title('Training Loss')
        plt.xlabel('Steps')
        plt.ylabel('Loss')
        plt.grid(True, alpha=0.3)
    
    if eval_loss:
        plt.subplot(1, 2, 2)
        plt.plot(eval_loss, 'orange')
        plt.title('Evaluation Loss')
        plt.xlabel('Epochs')
        plt.ylabel('Loss')
        plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print final metrics
    if train_loss:
        print(f"Final training loss: {train_loss[-1]:.4f}")
    if eval_loss:
        print(f"Final eval loss: {eval_loss[-1]:.4f}")
        if len(eval_loss) > 1:
            improvement = ((eval_loss[0] - eval_loss[-1]) / eval_loss[0]) * 100
            print(f"Eval loss improvement: {improvement:.1f}%")

## Test Generation

In [ ]:
# Test the trained model
def test_model_generation():
    test_prompt = """# Tools
<tools>
climate_control: Control vehicle climate settings
window_control: Control vehicle windows
seat_control: Control vehicle seat settings
lighting_control: Control vehicle lighting
drive_mode: Control vehicle drive mode
</tools>

User: Set the temperature to 72 degrees
Assistant:"""
    
    inputs = tokenizer(test_prompt, return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    model.eval()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id
        )
    
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print("Model output:")
    print("-" * 40)
    print(response)
    print("-" * 40)
    
    # Validate format
    has_tool_tags = "<tool_call>" in response and "</tool_call>" in response
    has_json = "{" in response and "}" in response
    has_correct_tool = "climate_control" in response
    
    print("\nFormat validation:")
    print(f"Tool call tags: {'YES' if has_tool_tags else 'NO'}")
    print(f"JSON structure: {'YES' if has_json else 'NO'}")
    print(f"Correct tool:   {'YES' if has_correct_tool else 'NO'}")
    
    return response

test_response = test_model_generation()

## Export Model

In [ ]:
# Merge LoRA weights and save
merged_model = model.merge_and_unload()
merged_model.save_pretrained("./qwen3-function-calling-merged")
tokenizer.save_pretrained("./qwen3-function-calling-merged")

print("Model saved to ./qwen3-function-calling-merged")
print("\nFor GGUF conversion run:")
print("python convert-hf-to-gguf.py ./qwen3-function-calling-merged --outtype q4_k_m")

## Evaluation with Strands Agents

Start llama.cpp server first:
```bash
./llama-server -m qwen3-finetuned.gguf --host 0.0.0.0 --port 8080 -c 2048 --jinja
```

In [ ]:
# Import production tools for evaluation
import sys
import requests
sys.path.append('../..')  # Add project root

from strands import Agent
from strands.models import LlamaCppModel
from src.agents.cockpit import (
    climate_control,
    window_control,
    seat_control,
    lighting_control,
    drive_mode
)

PRODUCTION_TOOLS = [
    climate_control,
    window_control,
    seat_control,
    lighting_control,
    drive_mode
]

# Check if server is running
def check_server(port):
    try:
        response = requests.get(f"http://localhost:{port}/health", timeout=1)
        return response.status_code == 200
    except:
        return False

if check_server(8080):
    print("Server running on port 8080")
else:
    print("Start llama.cpp server on port 8080 first")

In [ ]:
# Evaluate with Strands Agent
def evaluate_with_strands(port=8080):
    if not check_server(port):
        print(f"Server not running on port {port}")
        return
    
    # Create model and agent
    model = LlamaCppModel(
        base_url=f"http://localhost:{port}",
        params={"temperature": 0.7, "max_tokens": 200}
    )
    
    agent = Agent(
        model=model,
        tools=PRODUCTION_TOOLS,
        system_prompt="You are a vehicle assistant with access to control tools."
    )
    
    # Test cases
    test_cases = [
        ("Set temperature to 72", "climate_control"),
        ("Open driver window", "window_control"),
        ("Switch to sport mode", "drive_mode"),
        ("Turn on headlights", "lighting_control"),
        ("Adjust seat forward", "seat_control")
    ]
    
    correct = 0
    for prompt, expected in test_cases:
        try:
            response = agent.run(prompt)
            response_str = str(response)
            
            # Check if correct tool was called
            tool_called = None
            for tool in PRODUCTION_TOOLS:
                if tool.__name__ in response_str:
                    tool_called = tool.__name__
                    break
            
            is_correct = tool_called == expected
            if is_correct:
                correct += 1
            
            status = "PASS" if is_correct else "FAIL"
            print(f"{status}: {prompt:30} -> {expected}")
            
        except Exception as e:
            print(f"ERROR: {prompt:30} -> {str(e)[:50]}")
    
    accuracy = (correct / len(test_cases)) * 100
    print(f"\nAccuracy: {accuracy:.1f}% ({correct}/{len(test_cases)})")

# Run evaluation if server is available
if check_server(8080):
    evaluate_with_strands(8080)